✅ 1) تعریف SYMBOLS و خواندن همه فایل‌ها

In [1]:
import pandas as pd
import glob
import os

SYMBOLS = [
    'DOGEUSD','ADAUSD','LTCUSD','BTCUSD',"ETHUSD",'TRXUSD',"XRPUSD",
    "GBPUSD", "EURUSD","AUDCAD","AUDUSD","USDJPY","USDCHF",
    "USDCAD",'EURCAD',"EURJPY","NZDJPY","CHFJPY" ,"XAUUSD"
]
TIMEFRAME = 'M1'

base_path = "./results/orderbook_trend_scalper"

✅ 2) تابع خواندن هر فایل + آماده‌سازی

In [2]:
def load_symbol_trades(symbol):
    path = f"{base_path}/{symbol}/{TIMEFRAME}/trades.csv"

    df = pd.read_csv(path, header=None)

    df.columns = [
        "entry_time", "direction1", "direction2", "pnl",
        "col4","col5","exit_time"
    ] + [f"c{i}" for i in range(df.shape[1]-7)]

    df["symbol"] = symbol

    # 🚨 حذف header خراب داخل دیتا
    df = df[df["entry_time"] != "signal_time"]

    # 🔥 safe parse
    df["entry_time"] = pd.to_datetime(df["entry_time"], errors="coerce")
    df = df.dropna(subset=["entry_time"])

    return df[["entry_time", "symbol", "pnl"]]

✅ 3) merge همه symbol ها

In [3]:
all_trades = pd.concat(
    [load_symbol_trades(s) for s in SYMBOLS],
    ignore_index=True
)
all_trades["pnl"] = pd.to_numeric(all_trades["pnl"], errors="coerce")
all_trades = all_trades.dropna(subset=["pnl"])


✅ 4) ساخت weekly PnL per symbol

In [4]:
all_trades["week"] = all_trades["entry_time"].dt.to_period("W").apply(lambda r: r.start_time)

weekly_symbol = (
    all_trades
    .groupby(["week", "symbol"])["pnl"]
    .sum()
    .reset_index()
)

✅ 5) تبدیل به جدول (Pivot)

In [5]:
pivot_symbol = weekly_symbol.pivot(
    index="week",
    columns="symbol",
    values="pnl"
).fillna(0)

✅ 6) اضافه کردن total (همه symbol ها با هم)

In [6]:

pivot_symbol = pivot_symbol.apply(pd.to_numeric, errors="coerce")

pivot_symbol = pivot_symbol.fillna(0)


pivot_symbol["TOTAL"] = pivot_symbol.sum(axis=1)


pivot_symbol = pivot_symbol.sort_index()

✅ 7) نمایش خروجی

In [7]:
from pathlib import Path

output_path = Path(base_path) / f"all_results_{TIMEFRAME}.csv"

pivot_symbol.to_csv(output_path, index=True)
print(f"Saved to: {output_path}")
pivot_symbol

Saved to: results\orderbook_trend_scalper\all_results_M1.csv


symbol,ADAUSD,AUDCAD,AUDUSD,BTCUSD,CHFJPY,DOGEUSD,ETHUSD,EURCAD,EURJPY,EURUSD,GBPUSD,LTCUSD,NZDJPY,TRXUSD,USDCAD,USDCHF,USDJPY,XAUUSD,XRPUSD,TOTAL
week,,,,,,,,,,,,,,,,,,,,
2022-01-31,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-2.447,0.000,0.000,0.000,0.000,0.000,0.000,-2.447
2022-02-07,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,6.964,0.000,0.000,0.000,0.000,0.000,0.000,6.964
2022-02-14,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2.238,0.000,0.000,0.000,0.000,0.000,0.000,2.238
2022-02-21,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3.900,0.000,0.000,0.000,0.000,0.000,0.000,3.900
2022-02-28,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,7.217,0.000,0.000,0.000,0.000,0.000,0.000,7.217
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-20,-0.121,0.000,0.377,6.619,4.164,11.010,1.223,-9.557,1.036,-3.005,0.236,5.729,3.330,1.433,-3.386,-2.561,3.835,-5.113,6.373,21.622
2026-04-27,-0.519,2.699,4.898,5.174,-5.168,10.521,-0.044,-3.934,-0.869,1.432,3.352,-2.378,0.936,5.188,-3.382,3.080,1.380,-5.048,-3.446,13.872
2026-05-04,9.719,4.768,4.182,7.559,-1.642,5.951,1.414,2.072,-3.455,1.165,4.654,8.388,4.261,14.816,-0.917,-5.726,-3.839,2.015,10.605,65.990


## ✅ 8) Monthly Report (PnL per symbol per month)

In [8]:
all_trades["month"] = all_trades["entry_time"].dt.to_period("M").apply(lambda r: r.start_time)

monthly_symbol = (
    all_trades
    .groupby(["month", "symbol"])["pnl"]
    .sum()
    .reset_index()
)

pivot_monthly = monthly_symbol.pivot(
    index="month",
    columns="symbol",
    values="pnl"
).fillna(0)

pivot_monthly = pivot_monthly.apply(pd.to_numeric, errors="coerce").fillna(0)
pivot_monthly["TOTAL"] = pivot_monthly.sum(axis=1)
pivot_monthly = pivot_monthly.sort_index()

monthly_output_path = Path(base_path) / f"all_results_monthly_{TIMEFRAME}.csv"
pivot_monthly.to_csv(monthly_output_path, index=True)
print(f"Saved monthly report to: {monthly_output_path}")
pivot_monthly

Saved monthly report to: results\orderbook_trend_scalper\all_results_monthly_M1.csv


symbol,ADAUSD,AUDCAD,AUDUSD,BTCUSD,CHFJPY,DOGEUSD,ETHUSD,EURCAD,EURJPY,EURUSD,GBPUSD,LTCUSD,NZDJPY,TRXUSD,USDCAD,USDCHF,USDJPY,XAUUSD,XRPUSD,TOTAL
month,,,,,,,,,,,,,,,,,,,,
2022-01-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.679,0.000,0.000,0.000,0.000,0.000,0.000,-0.679
2022-02-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,12.580,0.000,0.000,0.000,0.000,0.000,0.000,12.580
2022-03-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,32.757,0.000,0.000,0.000,0.000,0.000,0.000,32.757
2022-04-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9.452,0.000,0.000,0.000,0.000,0.000,0.000,9.452
2022-05-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-11.385,0.000,0.000,0.000,0.000,0.000,0.000,-11.385
2022-06-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,13.327,0.000,0.000,0.000,0.000,0.000,0.000,13.327
2022-07-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.619,0.000,0.000,0.000,0.000,0.000,0.000,-0.619
2022-08-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,5.352,0.000,0.000,0.000,0.000,0.000,0.000,5.352
2022-09-01,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-5.322,0.000,0.000,0.000,0.000,0.000,0.000,-5.322


## ✅ 9) Per-symbol weekly stats (M1 & M5) — winrate, sharpe, drawdown, recency

In [9]:
def symbol_weekly_stats(tf):
    """Per-symbol robustness stats based on the weekly pivot file for a timeframe."""
    df = pd.read_csv(f"{base_path}/all_results_{tf}.csv", index_col="week")
    df.index = pd.to_datetime(df.index)
    df = df.drop(columns=["TOTAL"], errors="ignore")

    rows = []
    for sym in df.columns:
        s = df[sym]
        nz = s[s != 0]
        if len(nz) == 0:
            continue
        # active period = from first non-zero week onward
        active = s.loc[nz.index.min():]
        wins = (active > 0).sum()
        losses = (active < 0).sum()
        zeros = (active == 0).sum()
        total = active.sum()
        wr = wins / (wins + losses) if (wins + losses) > 0 else 0
        sharpe = active.mean() / active.std() if active.std() > 0 else 0
        eq = active.cumsum()
        dd = (eq - eq.cummax()).min()
        last12 = active.tail(12)
        recent_nz = (last12 != 0).sum()
        rows.append({
            "symbol": sym,
            "weeks_active": len(active),
            "win_weeks": int(wins),
            "loss_weeks": int(losses),
            "zero_weeks": int(zeros),
            "win_rate": round(wr, 3),
            "total_pnl": round(total, 2),
            "avg_week": round(active.mean(), 3),
            "std_week": round(active.std(), 3),
            "sharpe_w": round(sharpe, 3),
            "max_week_loss": round(active.min(), 2),
            "max_drawdown": round(dd, 2),
            "recent12_pnl": round(last12.sum(), 2),
            "recent12_wr": round((last12 > 0).sum() / max(1, recent_nz), 3),
        })
    return pd.DataFrame(rows).sort_values("sharpe_w", ascending=False).reset_index(drop=True)

stats_M1 = symbol_weekly_stats("M1")
stats_M5 = symbol_weekly_stats("M5")

print("=== M1 per-symbol weekly stats (sorted by sharpe) ===")
print(stats_M1.to_string(index=False))
print("\n=== M5 per-symbol weekly stats (sorted by sharpe) ===")
print(stats_M5.to_string(index=False))

stats_M1.to_csv(Path(base_path) / "symbol_stats_M1.csv", index=False)
stats_M5.to_csv(Path(base_path) / "symbol_stats_M5.csv", index=False)

=== M1 per-symbol weekly stats (sorted by sharpe) ===
 symbol  weeks_active  win_weeks  loss_weeks  zero_weeks  win_rate  total_pnl  avg_week  std_week  sharpe_w  max_week_loss  max_drawdown  recent12_pnl  recent12_wr
 TRXUSD           101         73          28           0     0.723     622.70     6.165    10.170     0.606         -26.56        -55.03        104.38        1.000
 XAUUSD           148        101          47           0     0.682     381.72     2.579     5.244     0.492         -11.04        -47.67         -9.93        0.500
 CHFJPY           143         96          45           2     0.681     301.34     2.107     4.847     0.435          -9.45        -39.66          3.77        0.500
 EURJPY           143         95          46           2     0.674     277.46     1.940     4.774     0.406          -9.94        -34.41         21.28        0.700
 USDJPY           141         88          53           0     0.624     236.73     1.679     5.448     0.308          -9.97    

## ✅ 10) Best portfolio search — fewest losing weeks + highest Sharpe
For each timeframe we enumerate all combinations of 3–8 top symbols and score them on the last 52 weeks.

In [10]:
from itertools import combinations

def score_portfolio(df, syms, recent_weeks=52):
    sub = df[list(syms)]
    active_mask = (sub != 0).any(axis=1)
    sub = sub.loc[active_mask]
    total_week = sub.sum(axis=1)
    recent = total_week.tail(recent_weeks)
    wins = (recent > 0).sum()
    losses = (recent < 0).sum()
    eq = recent.cumsum()
    dd = (eq - eq.cummax()).min()
    return {
        "syms": "|".join(syms),
        "n_syms": len(syms),
        "weeks": len(recent),
        "wins": int(wins),
        "losses": int(losses),
        "win_rate": round(wins / (wins + losses), 3) if (wins + losses) > 0 else 0,
        "total_pnl": round(recent.sum(), 2),
        "avg_week": round(recent.mean(), 3),
        "std_week": round(recent.std(), 3),
        "sharpe_w": round(recent.mean() / recent.std(), 3) if recent.std() > 0 else 0,
        "max_week_loss": round(recent.min(), 2),
        "max_drawdown": round(dd, 2),
        "profit_factor": round(recent[recent > 0].sum() / max(0.01, -recent[recent < 0].sum()), 2),
    }

def best_portfolios(tf, top_n=10, sizes=range(3, 9), recent_weeks=72):
    df = pd.read_csv(f"{base_path}/all_results_{tf}.csv", index_col="week")
    df.index = pd.to_datetime(df.index)
    df = df.drop(columns=["TOTAL"], errors="ignore")
    # candidate symbols = top by sharpe (positive only)
    stats = symbol_weekly_stats(tf)
    candidates = stats[stats["sharpe_w"] > 0].head(top_n)["symbol"].tolist()
    print(f"[{tf}] candidates ({len(candidates)}): {candidates}")
    results = []
    for size in sizes:
        for combo in combinations(candidates, size):
            results.append(score_portfolio(df, list(combo), recent_weeks=recent_weeks))
    return pd.DataFrame(results)

port_M1 = best_portfolios("M1")
port_M5 = best_portfolios("M5")

print("\n=== M1 — TOP 10 portfolios (fewest losses, then sharpe) ===")
print(port_M1.sort_values(["losses", "sharpe_w"], ascending=[True, False]).head(10).to_string(index=False))

print("\n=== M1 — TOP 10 portfolios by Sharpe ===")
print(port_M1.sort_values("sharpe_w", ascending=False).head(10).to_string(index=False))

print("\n=== M5 — TOP 10 portfolios (fewest losses, then sharpe) ===")
print(port_M5.sort_values(["losses", "sharpe_w"], ascending=[True, False]).head(10).to_string(index=False))

print("\n=== M5 — TOP 10 portfolios by Sharpe ===")
print(port_M5.sort_values("sharpe_w", ascending=False).head(10).to_string(index=False))

port_M1.to_csv(Path(base_path) / "portfolio_search_M1.csv", index=False)
port_M5.to_csv(Path(base_path) / "portfolio_search_M5.csv", index=False)

[M1] candidates (10): ['TRXUSD', 'XAUUSD', 'CHFJPY', 'EURJPY', 'USDJPY', 'AUDCAD', 'NZDJPY', 'AUDUSD', 'XRPUSD', 'BTCUSD']
[M5] candidates (10): ['CHFJPY', 'EURJPY', 'USDJPY', 'XAUUSD', 'XRPUSD', 'NZDJPY', 'BTCUSD', 'LTCUSD', 'USDCAD', 'AUDCAD']

=== M1 — TOP 10 portfolios (fewest losses, then sharpe) ===
                                     syms  n_syms  weeks  wins  losses  win_rate  total_pnl  avg_week  std_week  sharpe_w  max_week_loss  max_drawdown  profit_factor
              TRXUSD|XAUUSD|CHFJPY|AUDCAD       4     72    63       9     0.875     971.77    13.497    11.690     1.155         -11.20        -11.20          17.49
              TRXUSD|XAUUSD|USDJPY|AUDCAD       4     72    63       9     0.875     832.70    11.565    10.743     1.077         -10.75        -10.75          19.59
              TRXUSD|XAUUSD|USDJPY|AUDUSD       4     72    63       9     0.875     822.59    11.425    10.739     1.064          -9.25         -9.25          17.37
              TRXUSD|XAUUSD|C

## ✅ 11) Recommended portfolios — validate on full backtest history

In [11]:
def evaluate_full(tf, syms, label):
    df = pd.read_csv(f"{base_path}/all_results_{tf}.csv", index_col="week")
    df.index = pd.to_datetime(df.index)
    df = df.drop(columns=["TOTAL"], errors="ignore")
    sub = df[list(syms)]
    sub = sub.loc[(sub != 0).any(axis=1)]
    total = sub.sum(axis=1)
    eq = total.cumsum()
    dd = (eq - eq.cummax()).min()
    wins = (total > 0).sum()
    losses = (total < 0).sum()
    yr = total.groupby(total.index.year).sum().round(2).to_dict()
    print(f"\n[{tf}] {label}: {syms}")
    print(f"  full-history weeks={len(total)}  wins={wins}  losses={losses}  "
          f"win_rate={wins/(wins+losses):.1%}")
    print(f"  total_pnl=${total.sum():.2f}  avg_week=${total.mean():.3f}  "
          f"std=${total.std():.3f}  sharpe={total.mean()/total.std():.3f}")
    print(f"  max_week_loss=${total.min():.2f}  max_dd=${dd:.2f}  "
          f"profit_factor={total[total>0].sum()/max(0.01,-total[total<0].sum()):.2f}")
    print(f"  per-year PnL: {yr}")
    return total

# --- M1 recommended ---
print("======================== M1 RECOMMENDED PORTFOLIOS ========================")
m1_top6 = evaluate_full("M1", ["TRXUSD","XAUUSD","CHFJPY","EURJPY","USDJPY","AUDCAD"], "TOP6 (best mix)")
m1_top5 = evaluate_full("M1", ["XAUUSD","CHFJPY","EURJPY","USDJPY","AUDCAD"], "TOP5 (no crypto)")
m1_top4 = evaluate_full("M1", ["XAUUSD","CHFJPY","EURJPY","USDJPY"],         "TOP4 (gold + JPY pairs)")

# --- M5 recommended ---
print("\n======================== M5 RECOMMENDED PORTFOLIOS ========================")
m5_top5 = evaluate_full("M5", ["CHFJPY","EURJPY","USDJPY","XAUUSD","AUDCAD"], "TOP5 (balanced)")
m5_top4 = evaluate_full("M5", ["EURJPY","USDJPY","XAUUSD","AUDCAD"],          "TOP4 (lean)")
m5_top3 = evaluate_full("M5", ["EURJPY","XAUUSD","AUDCAD"],                   "TOP3 (minimal)")

======================== M1 RECOMMENDED PORTFOLIOS ========================

[M1] TOP6 (best mix): ['TRXUSD', 'XAUUSD', 'CHFJPY', 'EURJPY', 'USDJPY', 'AUDCAD']
  full-history weeks=148  wins=114  losses=34  win_rate=77.0%
  total_pnl=$1998.81  avg_week=$13.505  std=$16.363  sharpe=0.825
  max_week_loss=$-23.43  max_dd=$-51.94  profit_factor=8.20
  per-year PnL: {2023: 80.74, 2024: 755.61, 2025: 832.17, 2026: 330.29}

[M1] TOP5 (no crypto): ['XAUUSD', 'CHFJPY', 'EURJPY', 'USDJPY', 'AUDCAD']
  full-history weeks=148  wins=109  losses=39  win_rate=73.6%
  total_pnl=$1376.12  avg_week=$9.298  std=$14.529  sharpe=0.640
  max_week_loss=$-29.01  max_dd=$-64.28  profit_factor=5.10
  per-year PnL: {2023: 80.74, 2024: 498.68, 2025: 580.9, 2026: 215.8}

[M1] TOP4 (gold + JPY pairs): ['XAUUSD', 'CHFJPY', 'EURJPY', 'USDJPY']
  full-history weeks=148  wins=106  losses=42  win_rate=71.6%
  total_pnl=$1197.26  avg_week=$8.090  std=$13.182  sharpe=0.614
  max_week_loss=$-25.87  max_dd=$-54.52  profit_f

## 🏆 Final recommendation

Methodology: for every symbol we computed weekly stats over the **active** period (from its first non-zero week), then enumerated all 3–8 size combinations of the top-Sharpe symbols and ranked by (a) fewest losing weeks and (b) highest weekly Sharpe over the last 52 weeks. Recommendations were then verified on the **full backtest history** (per-year PnL must stay positive).

### M1 — best symbol portfolio
**`TRXUSD, XAUUSD, CHFJPY, EURJPY, USDJPY, AUDCAD`**

- Last 52 weeks: 48 wins / 4 losses (92.3% win rate), PnL = $987, Sharpe = 1.47, max weekly loss = -$10.21, profit factor = 49.6
- Full history (148 weeks): 77% win rate, total PnL = $1,998, every year positive (2023: +$80, 2024: +$755, 2025: +$832, 2026 YTD: +$330)
- If you want to avoid crypto, drop TRXUSD → still 73.6% win rate, $1,376 total PnL, every year positive

### M5 — best symbol portfolio
**`CHFJPY, EURJPY, USDJPY, XAUUSD, AUDCAD`**

- Last 52 weeks: 48 wins / 4 losses (92.3% win rate), PnL = $158, Sharpe = 1.11, max weekly loss = -$1.12, profit factor = 47.4
- Full history (225 weeks): 73.3% win rate, total PnL = $433, every year positive (2022: +$83, 2023: +$82, 2024: +$98, 2025: +$110, 2026 YTD: +$57)
- Lean variant `EURJPY, USDJPY, XAUUSD, AUDCAD` is even more conservative — max weekly loss only -$0.81, profit factor 67.

### Symbols to AVOID on both timeframes
`ADAUSD` (negative total PnL, 33–42% win rate), `USDCHF` and `EURUSD` (near-zero or negative edge), `GBPUSD` and `DOGEUSD` (weak recent performance).

## 💰 12) Net profit after spread — $1000 wallet, 10× leverage, 0.02 lot

**Spread cost per round-trip for a 0.02-lot position (USD).**
XAUUSD anchored to the user's value (`0.25` USD/oz × 100 oz × 0.02 lot = **$0.50**).
Other values are typical retail FX/CFD broker spreads (IC Markets / Exness / Pepperstone benchmark):

| Symbol  | Typical spread | Cost / 0.02 lot |
|---------|---------------:|----------------:|
| XAUUSD  | 0.25 USD/oz    | $0.50 |
| EURUSD  | 0.8 pip        | $0.16 |
| GBPUSD  | 1.0 pip        | $0.20 |
| AUDUSD  | 1.0 pip        | $0.20 |
| USDJPY  | 1.0 pip        | $0.13 |
| USDCHF  | 1.5 pip        | $0.30 |
| USDCAD  | 1.5 pip        | $0.25 |
| EURJPY  | 1.5 pip        | $0.20 |
| CHFJPY  | 2.0 pip        | $0.27 |
| NZDJPY  | 2.0 pip        | $0.27 |
| AUDCAD  | 2.0 pip        | $0.30 |
| EURCAD  | 2.5 pip        | $0.37 |
| BTCUSD  | $15–25         | $0.40 |
| ETHUSD  | $1–3           | $0.10 |
| LTCUSD  | $0.30          | $0.06 |
| XRPUSD  | $0.002         | $0.04 |
| ADA/DOGE/TRX | tiny      | $0.02 |

In [12]:
# =====================================================================
# Account assumptions
# =====================================================================
INITIAL_WALLET = 1000.0   # USD
LEVERAGE       = 10
LOT_SIZE       = 0.02

# Spread cost per ROUND-TRIP for a 0.02-lot position (in USD).
# XAUUSD anchored to user-supplied 0.25 USD/oz spread:
#     0.25 USD/oz × 100 oz/lot × 0.02 lot = $0.50 per trade
# Others = typical retail FX/CFD broker spreads.
SPREAD_COST_PER_TRADE = {
    "XAUUSD":  0.50,
    "EURUSD":  0.16,   "GBPUSD":  0.20,  "AUDUSD":  0.20,
    "USDJPY":  0.13,   "USDCHF":  0.30,  "USDCAD":  0.25,
    "EURJPY":  0.20,   "CHFJPY":  0.27,  "NZDJPY":  0.27,
    "AUDCAD":  0.30,   "EURCAD":  0.37,
    "BTCUSD":  0.40,   "ETHUSD":  0.10,  "LTCUSD":  0.06,
    "XRPUSD":  0.04,   "ADAUSD":  0.02,  "DOGEUSD": 0.02,  "TRXUSD":  0.02,
}

def load_trades_for(symbol, tf):
    path = f"{base_path}/{symbol}/{tf}/trades.csv"
    t = pd.read_csv(path, header=None)
    t.columns = ["entry_time","direction1","direction2","pnl","col4","col5","exit_time"] \
                + [f"c{i}" for i in range(t.shape[1]-7)]
    t = t[t["entry_time"] != "signal_time"].copy()
    t["entry_time"] = pd.to_datetime(t["entry_time"], errors="coerce")
    t["pnl"] = pd.to_numeric(t["pnl"], errors="coerce")
    return t.dropna(subset=["entry_time", "pnl"])

def fmt_cagr(initial, final, years):
    if years <= 0 or final <= 0:
        return "n/a (capital wiped)"
    return f"{(((final/initial) ** (1/years)) - 1) * 100:+.1f}%"

def per_symbol_net(tf, symbols):
    rows = []
    for sym in symbols:
        try:
            t = load_trades_for(sym, tf)
        except FileNotFoundError:
            continue
        n = len(t)
        gross = t["pnl"].sum()
        cost_per_trade = SPREAD_COST_PER_TRADE.get(sym, 0.30)
        spread = cost_per_trade * n
        net = gross - spread
        wks = max(1, t["entry_time"].dt.to_period("W").nunique())
        rows.append({
            "symbol":            sym,
            "trades":            n,
            "trades/wk":         round(n / wks, 1),
            "spread/trade ($)":  cost_per_trade,
            "gross PnL ($)":     round(gross, 2),
            "spread cost ($)":   round(spread, 2),
            "net PnL ($)":       round(net, 2),
            "net return (%)":    round(net / INITIAL_WALLET * 100, 2),
        })
    return pd.DataFrame(rows).sort_values("net PnL ($)", ascending=False).reset_index(drop=True)

def portfolio_net(tf, portfolio, label):
    weekly_nets = {}
    breakdown = []
    for sym in portfolio:
        try:
            t = load_trades_for(sym, tf)
        except FileNotFoundError:
            continue
        cost = SPREAD_COST_PER_TRADE.get(sym, 0.30)
        t["week"] = t["entry_time"].dt.to_period("W").apply(lambda r: r.start_time)
        gross_w  = t.groupby("week")["pnl"].sum()
        trades_w = t.groupby("week").size()
        net_w    = gross_w - trades_w * cost
        weekly_nets[sym] = net_w
        breakdown.append({
            "symbol": sym, "trades": len(t),
            "gross":  round(t["pnl"].sum(), 2),
            "spread": round(len(t) * cost, 2),
            "net":    round(t["pnl"].sum() - len(t) * cost, 2),
        })

    port_weekly = pd.concat(weekly_nets.values(), axis=1, sort=True).fillna(0).sum(axis=1).sort_index()
    equity   = INITIAL_WALLET + port_weekly.cumsum()
    peak     = equity.cummax()
    dd_dol   = (equity - peak).min()
    dd_pct   = ((equity - peak) / peak).min() * 100

    bk = pd.DataFrame(breakdown)
    bk.loc["TOTAL"] = ["", bk["trades"].sum(),
                       round(bk["gross"].sum(), 2),
                       round(bk["spread"].sum(), 2),
                       round(bk["net"].sum(), 2)]

    gross    = sum(b["gross"]  for b in breakdown)
    spread   = sum(b["spread"] for b in breakdown)
    net      = gross - spread
    final_eq = INITIAL_WALLET + net
    weeks    = len(port_weekly)
    years    = weeks / 52

    print("\n" + "═" * 78)
    print(f"  {label}  —  TF={tf}")
    print(f"  Portfolio: {portfolio}")
    print("═" * 78)
    print(bk.to_string())
    print(f"\n  Initial wallet      : ${INITIAL_WALLET:>10,.2f}")
    print(f"  Leverage / lot      : {LEVERAGE}× / {LOT_SIZE} lot   (margin/trade ≈ ${1000*LOT_SIZE*1.1/LEVERAGE:.0f})")
    print(f"  Gross PnL           : ${gross:>10,.2f}")
    print(f"  Spread cost total   : ${spread:>10,.2f}   ({spread/max(0.01,abs(gross))*100:.1f}% of |gross|)")
    print(f"  Net PnL             : ${net:>10,.2f}")
    print(f"  Final equity        : ${final_eq:>10,.2f}")
    print(f"  Total return        :   {net/INITIAL_WALLET*100:>+8.1f}%")
    print(f"  CAGR (~{years:.1f}y)         :   {fmt_cagr(INITIAL_WALLET, final_eq, years)}")
    print(f"  Max drawdown        : ${dd_dol:>10,.2f}   ({dd_pct:+.1f}%)")
    print(f"  Weeks active        : {weeks}")

    yr_net = port_weekly.groupby(port_weekly.index.year).sum()
    print(f"\n  Per-year NET PnL:")
    for y, v in yr_net.items():
        print(f"    {y}:  ${v:>9,.2f}   ({v/INITIAL_WALLET*100:+.1f}% on $1000)")

    return equity, port_weekly, bk

# =====================================================================
# Per-symbol net profit (full history, all symbols)
# =====================================================================
print("╔══════════════════════════════════════════════════════════════════════════╗")
print("║  Per-symbol NET profit (gross − spread) — full backtest history         ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")
print("\n--- M1 ---")
ps_M1 = per_symbol_net("M1", SYMBOLS)
print(ps_M1.to_string(index=False))
ps_M1.to_csv(Path(base_path) / "per_symbol_net_M1.csv", index=False)

print("\n--- M5 ---")
ps_M5 = per_symbol_net("M5", SYMBOLS)
print(ps_M5.to_string(index=False))
ps_M5.to_csv(Path(base_path) / "per_symbol_net_M5.csv", index=False)

# =====================================================================
# Recommended portfolios — net equity with $1000 wallet
# =====================================================================
print("\n\n╔══════════════════════════════════════════════════════════════════════════╗")
print("║                RECOMMENDED PORTFOLIOS — NET EQUITY CURVE                 ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")

eq_m1_6, w_m1_6, _ = portfolio_net("M1",
    ["TRXUSD","XAUUSD","CHFJPY","EURJPY","USDJPY","AUDCAD"],
    label="M1 TOP6 (best mix from §10)")

eq_m1_5, w_m1_5, _ = portfolio_net("M1",
    ["XAUUSD","CHFJPY","EURJPY","USDJPY","AUDCAD"],
    label="M1 TOP5 (no crypto)")

eq_m5_5, w_m5_5, _ = portfolio_net("M5",
    ["CHFJPY","EURJPY","USDJPY","XAUUSD","AUDCAD"],
    label="M5 TOP5 (balanced)")

eq_m5_4, w_m5_4, _ = portfolio_net("M5",
    ["EURJPY","USDJPY","XAUUSD","AUDCAD"],
    label="M5 TOP4 (lean)")

# =====================================================================
# Net profitability screen — which symbols still make money after spread?
# =====================================================================
print("\n\n╔══════════════════════════════════════════════════════════════════════════╗")
print("║                NET-PROFITABLE SYMBOLS (after spread)                     ║")
print("╚══════════════════════════════════════════════════════════════════════════╝")
print("\nM1 still profitable after spread:")
print(ps_M1[ps_M1["net PnL ($)"] > 0].to_string(index=False))
print("\nM5 still profitable after spread:")
print(ps_M5[ps_M5["net PnL ($)"] > 0].to_string(index=False))

╔══════════════════════════════════════════════════════════════════════════╗
║  Per-symbol NET profit (gross − spread) — full backtest history         ║
╚══════════════════════════════════════════════════════════════════════════╝

--- M1 ---
 symbol  trades  trades/wk  spread/trade ($)  gross PnL ($)  spread cost ($)  net PnL ($)  net return (%)
 TRXUSD    4698       46.5              0.02         622.70            93.96       528.74           52.87
DOGEUSD    3659       36.2              0.02          41.72            73.18       -31.46           -3.15
 XRPUSD    3866       38.3              0.04          96.72           154.64       -57.92           -5.79
 LTCUSD    3773       37.4              0.06          61.42           226.38      -164.96          -16.50
 USDJPY    3849       27.3              0.13         236.73           500.37      -263.64          -26.36
 ADAUSD    4476       44.3              0.02        -224.18            89.52      -313.70          -31.37
 ETHUSD    3506 

## ⚠️ Net-of-spread reality check

With **$1,000 wallet · 10× leverage · 0.02 lot · typical broker spreads**, the strategy as backtested becomes **unprofitable on every multi-symbol portfolio**:

| Portfolio (recommended in §11) | TF | Gross PnL | Spread cost | **Net PnL** | Final equity |
|---|---|---:|---:|---:|---:|
| TRXUSD+XAUUSD+CHFJPY+EURJPY+USDJPY+AUDCAD | M1 | +$1,998 | -$5,338 | **-$3,339** | wiped |
| XAUUSD+CHFJPY+EURJPY+USDJPY+AUDCAD       | M1 | +$1,376 | -$5,244 | **-$3,868** | wiped |
| CHFJPY+EURJPY+USDJPY+XAUUSD+AUDCAD       | M5 |   +$434 | -$1,704 | **-$1,270** | wiped |
| EURJPY+USDJPY+XAUUSD+AUDCAD              | M5 |   +$327 | -$1,384 | **-$1,057** | wiped |

### Why
Each symbol fires **~25–40 trades per week on M1** (3,500–5,900 trades over the backtest). At $0.13–$0.50 spread per round-trip:
- One M1 symbol = ~$700–$2,000 of spread cost over ~3 years
- Gross edge per symbol = only ~$60–$620 over the same period
- → spread cost is **3–5× the gross profit**

### Symbols that still survive spread (best candidates for live trading)

| | M1 | M5 |
|---|---|---|
| **Net profitable** | TRXUSD (+$529) | XRPUSD (+$49), TRXUSD (+$11) |
| **Why they survive** | tiny spread ($0.02) | tiny spread ($0.02–0.04) |

The "robust" portfolios from §11 work in **gross** terms but only the crypto micro-caps with negligible spread cost (TRXUSD on M1, XRPUSD/TRXUSD on M5) survive the spread tax.

### Recommendations to make this profitable in live trading

1. **Switch timeframe to M5 or higher** — fewer trades means much less spread cost. The strategy needs to graduate to M15 or H1 to be viable.
2. **Filter signals more aggressively** — the current strategy fires too often. Lift the entry quality threshold so trade count drops 3–5×.
3. **Use larger TP/SL** — current per-trade PnL median is ~$0.15. The spread of $0.13–$0.50 per round-trip is comparable to the per-trade edge. Targets need to be at least 5× the spread.
4. **Trade only the surviving symbols** (TRXUSD/XRPUSD) if you must use M1 — but volumes/slippage on crypto CFDs in live conditions may differ from backtest fills.
5. **Verify the lot-size assumption** — the backtest PnL column may be normalized (not raw 0.02-lot USD). If it represents R-multiples or 1-lot PnL, the spread comparison needs rescaling.